In [1]:
!git clone https://github.com/AbdullahShheik/promptbert-reimplementation.git
%cd promptbert-reimplementation

fatal: destination path 'promptbert-reimplementation' already exists and is not an empty directory.
/content/promptbert-reimplementation


In [2]:
!pip install torch==2.6.0 torchvision==0.21.0 scipy==1.15.0 huggingface-hub==0.26.2 numpy==1.26.4 transformers==4.38.0 tokenizers==0.15.2

In [3]:
!cd SentEval/data/downstream && bash download_dataset.sh

/bin/bash: line 1: cd: SentEval/data/downstream: No such file or directory


In [4]:
%%writefile ensemble_eval.py
"""Multi-template ensembling for prompt-based sentence embeddings (PromptBERT).
Mirrors the prompt branch of evaluation.py: same preprocessing, same [MASK] extraction,
same SentEval scoring. Only change: average [MASK] vectors over several templates.

Usage:
  python ensemble_eval.py <model> <template_set> "<variants>"
  e.g. python ensemble_eval.py bert-base-uncased bert "T1;T1+T2+T3+T4"
       python ensemble_eval.py royokong/unsup-PromptRoBERTa roberta "T1;T1+T2"
"""
import sys, time
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
sys.path.insert(0, './SentEval')
import senteval

MODEL, TSET, VARIANTS = sys.argv[1], sys.argv[2], sys.argv[3].split(';')
TASKS = ['STS12', 'STS13', 'STS14', 'STS15', 'STS16', 'STSBenchmark', 'SICKRelatedness']

# {s} = sentence, {m} = the tokenizer's mask token ([MASK] for BERT, <mask> for RoBERTa).
TEMPLATE_SETS = {
    # Unfine-tuned BERT. T1 = run.sh bert-prompt template; T2-T4 = other Table 4 rows.
    'bert': {
        'T1': 'This sentence : "{s}" means{m}.',
        'T2': 'This sentence of "{s}" means{m}.',
        'T3': 'This sentence of {s} means{m}.',
        'T4': 'This {s} means{m}.',
    },
    # unsup-PromptRoBERTa. T1 = eval_only.sh unsup-roberta template;
    # T2 = its second training template (paper Table 13); T3-T4 = unseen in training.
    'roberta': {
        'T1': "This sentence : ' {s} ' means{m}.",
        'T2': "The sentence : ' {s} ' means{m}.",
        'T3': "This sentence of ' {s} ' means{m}.",
        'T4': 'This {s} means{m}.',
    },
}
TEMPLATES = TEMPLATE_SETS[TSET]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
model = AutoModel.from_pretrained(MODEL).to(device).eval()
cache = {k: {} for k in TEMPLATES}   # each template's embedding is computed once and reused
print(f'model={MODEL} templates={TSET} device={device}', flush=True)


def prep(words):
    s = ' '.join(words)
    if len(s) > 0 and s[-1] not in '.?"\'':   # same rule as evaluation.py
        s += '.'
    return s


@torch.no_grad()
def embed(t, sents):
    todo = [s for s in dict.fromkeys(sents) if s not in cache[t]]
    for i in range(0, len(todo), 128):
        chunk = todo[i:i + 128]
        texts = [TEMPLATES[t].replace('{m}', tok.mask_token).replace('{s}', s).strip() for s in chunk]
        b = tok(texts, return_tensors='pt', padding=True).to(device)
        h = model(**b).last_hidden_state
        vec = h[b['input_ids'] == tok.mask_token_id]
        assert vec.shape[0] == len(chunk), f'{t}: expected one mask token per sentence'
        for s, v in zip(chunk, vec.cpu().numpy()):
            cache[t][s] = v
    if todo:
        print(f'  [{t}] {len(cache[t])} sentences embedded so far', flush=True)
    return np.stack([cache[t][s] for s in sents])


def make_batcher(variant):
    names = variant.split('+')
    def batcher(params, batch):
        if batch and batch[0] and isinstance(batch[0][0], bytes):
            batch = [[w.decode('utf-8') for w in s] for s in batch]
        sents = [prep(s) for s in batch]
        vs = [embed(n, sents) for n in names]
        return np.mean(vs, axis=0)
    return batcher


params = {'task_path': './SentEval/data', 'usepytorch': True, 'kfold': 10}
rows = []
for var in VARIANTS:
    t0 = time.time()
    print(f'=== {var} ===', flush=True)
    r = senteval.engine.SE(params, make_batcher(var), lambda p, s: None).eval(TASKS)
    sc = [r[t]['all']['spearman']['all'] * 100 for t in TASKS[:5]]
    sc += [r['STSBenchmark']['test']['spearman'][0] * 100, r['SICKRelatedness']['test']['spearman'][0] * 100]
    dev = r['STSBenchmark']['dev']['spearman'][0] * 100
    rows.append((var, dev, sc))
    print(f'{var}: STSB-dev {dev:.2f} | test avg {np.mean(sc):.2f} ({time.time() - t0:.0f}s)', flush=True)

print(f'\nModel: {MODEL}')
print('| Variant | STSB-dev | ' + ' | '.join(TASKS) + ' | Avg. |')
for var, dev, sc in rows:
    print(f'| {var} | {dev:.2f} | ' + ' | '.join(f'{x:.2f}' for x in sc) + f' | {np.mean(sc):.2f} |')


Overwriting ensemble_eval.py


In [12]:
!python ensemble_eval.py bert-base-uncased bert "T1;T2;T3;T4;T1+T2;T1+T2+T3;T1+T2+T3+T4"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
model=bert-base-uncased templates=bert device=cuda
=== T1 ===
  [T1] 128 sentences embedded so far
  [T1] 256 sentences embedded so far
  [T1] 384 sentences embedded so far
  [T1] 511 sentences embedded so far
  [T1] 639 sentences embedded so far
  [T1] 766 sentences embedded so far
  [T1] 894 sentences embedded so far
  [T1] 1017 sentences embedded so far
  [T1] 1145 sentences embedded so far
  [T1] 1270 sentences embedded so far
  [T1] 1379 sentences embedded so far
  [T1] 1487 sentences embedded so far
  [T1] 1600 sentences embedded so far
  [T1] 1706 sentences embedded so far
  [T1] 1812 sentences embedded so far
  [T1] 1918 sentences embedded so far
  [T1] 1994 sentences embedded so far
  [T1] 2077 sent

In [10]:
!python ensemble_eval.py royokong/unsup-PromptRoBERTa roberta "T1;T2;T3;T4;T1+T2;T1+T2+T3;T1+T2+T3+T4"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100% 653/653 [00:00<00:00, 5.03MB/s]
vocab.json: 798kB [00:00, 31.6MB/s]
merges.txt: 456kB [00:00, 33.9MB/s]
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
pytorch_model.bin: 100% 501M/501M [00:10<00:00, 48.6MB/s]
model=royokong/unsup-PromptRoBERTa templates=roberta device=cuda
=== T1 ===
  [T1] 128 sentences embedded so far
  [T1] 256 sentences embedded so far
  [T1] 384 sentences embedded so far
  [T1] 511 sentences embedded so far
  [T1] 639 sentences embedded 